In [6]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

In [10]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered.en.subword.train
        path_tgt: en-zh.zh-filtered.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered.en.subword.dev
        path_tgt: en-zh.zh-filtered.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [ ]:
# Find the number of CPUs/cores on the machine
!nproc --all

In [3]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 7

Corpus corpus_1's weight should be given. We default it to 1 for you.
[2025-03-21 01:38:12,956 INFO] Counter vocab from -1 samples.
[2025-03-21 01:38:12,956 INFO] n_sample=-1: Build vocab on full datasets.
[2025-03-21 01:38:16,602 INFO] Counters src: 4594
[2025-03-21 01:38:16,602 INFO] Counters tgt: 2820


In [2]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-3da117a9-dc75-64e3-3c01-4f5d3b83c278)


In [3]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)

True
NVIDIA A100-SXM4-40GB
Free GPU memory: 39900.25 out of: 40326.375


In [11]:
# Train the NMT model
!onmt_train -config config.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/venv/main/l

## Translate

In [12]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
# gpu
# !onmt_translate -model models/model.step-9000-bleu-0.1.pt -src en-zh.en-filtered.en.subword.test -output zh.translated -gpu 0 -min_length 1

!onmt_translate \
  -model models/model.step-9000-bleu-0.1.pt \
  -src en-zh.en-filtered.en.subword.test \
  -output zh.translated \
  -gpu -1 \
  -min_length 1


[2025-03-30 04:22:12,625 INFO] Loading checkpoint from models/model.step-9000-bleu-0.1.pt
[2025-03-30 04:22:14,622 INFO] Loading data into the model
[2025-03-30 04:30:07,871 INFO] PRED SCORE: -0.5437, PRED PPL: 1.72 NB SENTENCES: 2000
Time w/o python interpreter load/terminate:  475.27239203453064


In [16]:
%pip install "numpy<2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 92.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.2
    Uninstalling numpy-2.1.2:
      Successfully uninstalled numpy-2.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.21.0+cu124 requires torch==2.6.0, but you have torch 2.2.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [13]:
# Check the first 5 lines of the translation file
!head -n 30 zh.translated

▁ 非洲 医疗 保险 和 健康 研究 的 ▁ 越来越 糟 , ▁需要 一个 新 眼 光 。 我们 不能 一直 ▁以 自己 的方式 做 这些事情 。
▁ 不好意思 。
▁这是 美国 最 著名的 医院 之一
▁然后 把 旅行 变成 游戏 , ▁ 一 路上 都有 意 想 不到 的 转 折 。
▁这是个 神奇的 装置 。
▁ 谢谢大家 的 关注 。
▁你 有一个 梦想 , ▁你 的 前 方 也有 障碍 , 我们都 一样 。
▁这个 城镇 欣 然 接受 它 。
▁我们 特别 受到 花 朵 的 触 动 , ▁我们 很 好奇 花 是怎么 摆 到 那里 的 。
▁我们也 和 来自 宾 州 大学 的 科学家 和 工程师 合作 , ▁ 开发 出 化学 驱动 版本 的 ▁ 变形 虫 机器人 。
▁所以我 借 用了 这个系统 来 在 1 9 9 0 年 1 1 月 在 巴黎 附近 的 凡 尔 赛 做 贸易 展览 。
▁它 太 痛苦 了 , 你 不想 去 想 它 。
▁我 看见 我 的名字 被 亮 着 了 , ▁因为 孩子们 把我 的名字 放到 了 灯光 里 。
▁它们 使你 能够 机械 地 呼吸 , ▁或者 机 动 地 呼吸 。
▁我说 :“ 什么 错了 ?” 他说 :“ 有个 问题 , 先生 。”
▁在 混乱 的 中心 , 我在 轮椅 上 滚 动 , ▁我 完全 是 隐 形 的 , 完全 是 隐 形 的 。
▁ 我们想要 将 这一 地区的 有毒 土壤 清理 干净 , ▁并 有一个 有机 花园 。
▁ 到 7 0 年代 , ▁ 家庭 烹 饪 却 处于 一种 悲 哀 的状态 , ▁ 高 脂 肪 和 香 料 食品 , ▁例如 麦克 纳 吉 特 和 惠 特 克 这样的 食品 , ▁我们都 拥有 最 爱 的 , ▁事实上 , 让 这些 食品 ▁比 家里 人 更 让人 更有 吸引力 。
▁美国 的 人均 二氧化碳 排放量 为 1 7 . 5 公 吨
▁让我们 把这个 原则 应用 到 西 伯 利亚 。
▁现在 的问题是 , 我们 究竟 能 达成 多少 目标 ?
▁这是 什么 ?
▁我们对 故事 的 了解 越多 , ▁ 就越 有 地方 色彩 和 质 感 , ▁人们 就越 开始 感受到 角色 , ▁ 就越 感到 可 信 赖 , 而不是 更少 。
▁ 穷 国 需要 援助 。
▁ 曲线 具有 贝 塞 尔 公式

In [14]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated

Done desubwording! Output: zh.translated.desubword


In [19]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.en-filtered.en.subword.test

# Desubword the test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered.en.subword.test

Done desubwording! Output: en-zh.en-filtered.en.subword.test.desubword
Done desubwording! Output: en-zh.en-filtered.en.subword.test.desubword


In [16]:
# Check the first 5 lines of the desubworded translation file
!head -n 30 zh.translated.desubword

print("---------------")
# Check the first 5 lines of the desubworded reference
!head -n 30 en-zh.zh-filtered.zh.subword.test.desubword

非洲医疗保险和健康研究的 越来越糟, 需要一个新眼光。我们不能一直 以自己的方式做这些事情。
不好意思。
这是美国最著名的医院之一
然后把旅行变成游戏, 一路上都有意想不到的转折。
这是个神奇的装置。
谢谢大家的关注。
你有一个梦想, 你的前方也有障碍,我们都一样。
这个城镇欣然接受它。
我们特别受到花朵的触动, 我们很好奇花是怎么摆到那里的。
我们也和来自宾州大学的科学家和工程师合作, 开发出化学驱动版本的 变形虫机器人。
所以我借用了这个系统来在1990年11月在巴黎附近的凡尔赛做贸易展览。
它太痛苦了,你不想去想它。
我看见我的名字被亮着了, 因为孩子们把我的名字放到了灯光里。
它们使你能够机械地呼吸, 或者机动地呼吸。
我说:“什么错了?”他说:“有个问题,先生。”
在混乱的中心,我在轮椅上滚动, 我完全是隐形的,完全是隐形的。
我们想要将这一地区的有毒土壤清理干净, 并有一个有机花园。
到70年代, 家庭烹饪却处于一种悲哀的状态, 高脂肪和香料食品, 例如麦克纳吉特和惠特克这样的食品, 我们都拥有最爱的, 事实上,让这些食品 比家里人更让人更有吸引力。
美国的人均二氧化碳排放量为17.5公吨
让我们把这个原则应用到西伯利亚。
现在的问题是,我们究竟能达成多少目标?
这是什么?
我们对故事的了解越多, 就越有地方色彩和质感, 人们就越开始感受到角色, 就越感到可信赖,而不是更少。
穷国需要援助。
曲线具有贝塞尔公式施加的 数学平滑。
我不知道他们是否会决定 是否写这份荣誉准则。
比如,如果任何人任何时候都能变成这样呢?
现在,还有一些其他规则 更多的是关于习惯和自然的。
取而代之的是,宏观层面和微级和平建设都需要 保持稳定,本地非政府机构, 当地政府和民间团体代表 必须是自下而上的主要角色。
当我们谈起权利的转变, 我们常常讨论亚洲的崛起。
---------------
非洲越来越低的医疗保健指数和越来越少人关注的医疗研究状况 必须改变,我们不能一直 停滞不前
很抱歉.
看看这幅画吧, 这是美国最著名的医院之一,
就把旅行变成游戏 一路上都有意想不到的惊奇
这个是很棒的机器。
谢谢大家的关注。
你有一个梦想, 你的前方有障碍,我们都一样。
这个城镇欣然接受了它。
特别是那些花,它们尤其让我们感动。 我们很好奇,那些花是怎么摆到那里的呢?”
我们还和来自宾州

## Evaluation

In [17]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-03-30 04:32:28--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 957 [text/plain]
Saving to: 'compute-bleu.py.1'

compute-bleu.py.1   100%[===================>]     957  --.-KB/s    in 0s      

2025-03-30 04:32:28 (53.7 MB/s) - 'compute-bleu.py.1' saved [957/957]



In [33]:
# Install sacrebleu
!pip3 install sacrebleu

In [20]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered.zh.subword.test.desubword zh.translated.desubword

Reference 1st sentence: 非洲越来越低的医疗保健指数和越来越少人关注的医疗研究状况 必须改变,我们不能一直 停滞不前
MTed 1st sentence: 非洲医疗保险和健康研究的 越来越糟, 需要一个新眼光。我们不能一直 以自己的方式做这些事情。
BLEU:  16.24320657978507
